In [18]:
import json
import pandas as pd
import os
from urllib.request import urlretrieve
from pydub import AudioSegment
from tqdm import tqdm

# Option 1: Parse JSON File

In [27]:
# json_path = "/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/annotation-project-17d6a079-0e06-4f72-96fb-6fab723ad827.json"
# json_path = '/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lit_caerulea/annotation-project-d48ca48d-1071-40fa-b3ae-2f73c997c87a.json'
json_path = '/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lit_peronii/Lit_peronii_soundbox_labels.json'

with open(json_path, 'r') as file:
    data = json.load(file)

data

{'uuid': 'a4b2a567-5bbf-4e69-aab3-bea9ddea2e0a',
 'name': '03_Litoria peronii',
 'description': 'Labelling Litoria peronii for FrogID. Added 15/08/2025.',
 'users': [{'id': '3f471540-d140-4a10-8541-199108c9a0a7',
   'username': 'Grace.Gillard',
   'email': 'Grace.Gillard@austmus.gov.au',
   'name': 'Grace Gillard'},
  {'id': '422b389b-840b-4ddb-afe6-65e0c22abe0b',
   'username': 'Andrew.Trevor-Jones',
   'email': 'Andrew.Trevor-Jones@austmus.gov.au',
   'name': 'Andrew Trevor-Jones'}],
 'tags': [{'id': 5, 'key': 'validated_status', 'value': 'published'},
  {'id': 6, 'key': 'validated_type', 'value': 'frogs'},
  {'id': 7, 'key': 'pending_status', 'value': 'submitted'},
  {'id': 8, 'key': 'geoprivacy', 'value': 'open'},
  {'id': 13, 'key': 'validated_by', 'value': 'Maureen Thompson'},
  {'id': 15, 'key': 'image_urls', 'value': '[]'},
  {'id': 16, 'key': 'quality_call', 'value': 'True'},
  {'id': 17, 'key': 'null_record', 'value': 'False'},
  {'id': 18, 'key': 'is_bam', 'value': 'False'},

In [28]:
# Map recording uuid to capture ID
recordings = data['recordings']
recordings_df = pd.DataFrame.from_dict(recordings)
uuid_to_capture_map = recordings_df.set_index('uuid')['path'].to_dict()
uuid_to_capture_map = {k: v.split('.')[0] for k,v in uuid_to_capture_map.items()}
uuid_to_capture_map

{'afba5511-1357-4461-ae0b-31f1e8dde121': '40667',
 '0c580e33-481e-4267-bdfb-f0261d8e6d6e': '45813',
 '51c80e58-d7c0-4c29-87d1-01eef86124f2': '46763',
 '128447fa-a092-41a8-a231-2176df026dfc': '55841',
 '287d8e36-ebf4-4be8-9ac8-9c81210103c9': '66540',
 '6d4ee5e3-b45c-4a55-824e-d9bc9435efc1': '82769',
 '1740951e-8c6e-4e2c-9a19-8a2d6d769e3b': '85280',
 '7b2a04bd-51c3-4d72-87a0-685ddc37a50d': '86045',
 '10af57ac-55eb-4297-befb-16e599719ffe': '86683',
 '6c3e4df2-3bd1-4373-8f7a-d47e0a08c680': '87360',
 'da8bff58-e47c-4d2c-b088-345c14fbb380': '88677',
 'a7fda202-d9b8-4488-a54c-42249c0f31a2': '90076',
 'eeefd146-9496-43dd-b0e2-59fd30de6331': '91461',
 'b77090e6-c1e0-4f12-a071-522fdce710ed': '92045',
 'a08e667d-bfac-484a-b4bf-614b678c1a2b': '93677',
 'b0cdee70-7ce7-41b5-b39f-5546c17fa8a6': '96835',
 '7ccbcea7-cea4-442f-9881-728f17306036': '97481',
 'd5b2ef75-3bd3-4f48-ba1e-6cd33c874617': '101260',
 'fb2b5b98-5da3-4c98-a9ce-5b367acc9c17': '106238',
 'a72394fa-bceb-4c46-a821-2d4ff7c2146b': '110697

In [44]:
# Map recording uuid to recording ID
uuid_to_id_map = recordings_df.set_index('uuid')['id'].to_dict()
uuid_to_id_map

{'afba5511-1357-4461-ae0b-31f1e8dde121': 2470,
 '0c580e33-481e-4267-bdfb-f0261d8e6d6e': 2471,
 '51c80e58-d7c0-4c29-87d1-01eef86124f2': 2472,
 '128447fa-a092-41a8-a231-2176df026dfc': 2473,
 '287d8e36-ebf4-4be8-9ac8-9c81210103c9': 2474,
 '6d4ee5e3-b45c-4a55-824e-d9bc9435efc1': 2475,
 '1740951e-8c6e-4e2c-9a19-8a2d6d769e3b': 2476,
 '7b2a04bd-51c3-4d72-87a0-685ddc37a50d': 2477,
 '10af57ac-55eb-4297-befb-16e599719ffe': 2478,
 '6c3e4df2-3bd1-4373-8f7a-d47e0a08c680': 2479,
 'da8bff58-e47c-4d2c-b088-345c14fbb380': 2480,
 'a7fda202-d9b8-4488-a54c-42249c0f31a2': 2481,
 'eeefd146-9496-43dd-b0e2-59fd30de6331': 2482,
 'b77090e6-c1e0-4f12-a071-522fdce710ed': 2483,
 'a08e667d-bfac-484a-b4bf-614b678c1a2b': 2484,
 'b0cdee70-7ce7-41b5-b39f-5546c17fa8a6': 2485,
 '7ccbcea7-cea4-442f-9881-728f17306036': 2486,
 'd5b2ef75-3bd3-4f48-ba1e-6cd33c874617': 2487,
 'fb2b5b98-5da3-4c98-a9ce-5b367acc9c17': 2488,
 'a72394fa-bceb-4c46-a821-2d4ff7c2146b': 2489,
 '9b59af78-6170-4027-b578-8cc0f5839512': 2490,
 '48451767-9e

In [45]:
# Extract start/end times from sound events
sound_events = data['sound_events']
sound_events_df = pd.DataFrame.from_dict(sound_events)
try:
    sound_events_df["begin_time"] = sound_events_df["geometry"].apply(lambda x: x["coordinates"][0])
    sound_events_df["end_time"] = sound_events_df["geometry"].apply(lambda x: x["coordinates"][2])
except Exception:
    sound_events_df["begin_time"] = sound_events_df["geometry"].apply(lambda x: eval(x)["coordinates"][0])
    sound_events_df["end_time"] = sound_events_df["geometry"].apply(lambda x: eval(x)["coordinates"][2])

sound_events_df

,id,uuid,recording_id,geometry_type,geometry,created_on,begin_time,end_time
0,2273,607de690-5398-40f4-97cd-c50943379ea6,3858,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[1.1544346...",2025-08-20 23:59:09.767302,1.154435,2.813926
1,2274,f669ba8e-e544-484e-9cea-1d2d93c4d3d5,3858,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[9.9670085...",2025-08-20 23:59:30.492477,9.967009,11.255600
2,2275,d9ab64b5-d6b4-4e91-a457-66ace60550bd,3858,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[24.385375...",2025-08-20 23:59:55.569458,24.385375,26.239876
3,2276,6c19c546-9bfd-45a7-a94d-545452fa0cdb,3858,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[27.802931...",2025-08-21 00:00:09.038933,27.802931,29.141230
4,2277,843c26a6-ce30-4812-93a4-c09ff3c3516a,3822,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[0.0527016...",2025-08-21 00:05:18.122377,0.052702,1.269952
...,...,...,...,...,...,...,...,...
524,10311,61ffa873-eeb2-41f8-94bd-8112e4d528db,2501,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[30.020007...",2025-09-04 06:38:07.129709,30.020008,32.036523
525,10312,8100326b-d362-4f8d-95a3-bc88291856d4,2501,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[36.692138...",2025-09-04 06:38:30.533382,36.692139,38.492390
526,10313,e33a553d-0d90-4e21-9020-77531ebe8df4,2501,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[42.753051...",2025-09-04 06:38:48.298642,42.753052,44.483163
527,10314,6aa5b55a-c99a-41ca-bea4-f7aa99cf41d5,2501,BoundingBox,"{""type"":""BoundingBox"",""coordinates"":[51.346414...",2025-09-04 06:39:03.858649,51.346414,53.111596


## Convert to Raven Selection Tables

The selection tables are .txt files with only the following columns required: ``Begin Time (s)``, ``End Time (s)``, ``Annotation``.

In [47]:
selection_table_dir = "/home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables"
os.makedirs(selection_table_dir, exist_ok=True)

count_skipped = 0
for uuid, capture_id in uuid_to_capture_map.items():

    # Find sound events corresponding to this recording
    try:
        sound_events = sound_events_df[sound_events_df['recording'] == uuid]
    except Exception:
        recording_id = uuid_to_id_map[uuid]
        sound_events = sound_events_df[sound_events_df['recording_id'] == recording_id]

    if len(sound_events) == 0:
        count_skipped += 1
        continue
    
    output_path = os.path.join(selection_table_dir, f"{capture_id}_selection_table.txt")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("Begin Time (s)\tEnd Time (s)\tAnnotation\n")

    for _, row in sound_events.iterrows():
        with open(output_path, "a", encoding="utf-8") as f:
            f.write(f"{row['begin_time']}\t{row['end_time']}\tevent\n")

print(f"Finished writing {len(uuid_to_capture_map) - count_skipped} selection tables to: {selection_table_dir}")

Finished writing 64 selection tables to: /home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables


## Create Info Table

Create a CSV file with 3 columns:

* ``fn``: Unique filename associated with each audio file
* ``audio_fp``: Filepaths to audio files in train set
* ``selection_table_fp``: Filepath to Raven selection tables

In [49]:
# Load captures table
captures_path = "/home/admin/julia-dev/data/frogid_captures/data_tables/FrogID_Captures_2025-03-09T21-01-13+0000.csv"
captures_df = pd.read_csv(captures_path)

/tmp/ipykernel_3386029/3616435119.py:3: DtypeWarning: Columns (32,34,35,39) have mixed types. Specify dtype option on import or set low_memory=False.
  captures_df = pd.read_csv(captures_path)


In [50]:
# Download audio files
def download_audio(audio_dir, capture_id, audio_url):
    save_path = os.path.join(audio_dir, f'{capture_id}.aac')
    urlretrieve(audio_url, save_path)
    audio = AudioSegment.from_file(save_path)
    wav_path = os.path.join(audio_dir, f'{capture_id}.wav')
    audio.export(wav_path, format='wav')
    os.remove(save_path)


audio_dir = "/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lit_peronii/audio"

captures_labelled = [x.split('_')[0] for x in os.listdir(selection_table_dir)]
for capture_id in tqdm(captures_labelled, desc="Downloading audio"):
    row = captures_df[captures_df['id'] == int(capture_id)]
    audio_url = row['call_audio'].values[0]
    download_audio(audio_dir, capture_id, audio_url)

In [51]:
# Generate pandas dataframe
info_table = []
for capture_id in captures_labelled:
    info_table.append({
        'fn': capture_id,
        'audio_fp': os.path.join(audio_dir, f'{capture_id}.wav'),
        'selection_table_fp': os.path.join(selection_table_dir, f'{capture_id}_selection_table.txt')
    })

info_df = pd.DataFrame.from_dict(info_table)

# Export as CSV
csv_out = 'datasets/frogid/Lit_peronii_info.csv'
info_df.to_csv(csv_out)
print(f'Saved info table to: {csv_out}')

Saved info table to: datasets/frogid/Lit_peronii_info.csv


In [ ]:
# import os
# import pandas as pd

# audio_dir = "/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/audio"
# selection_table_dir = "/home/admin/julia-dev/voxaboxen/datasets/frogid/Lim_peronii_formatted/Lim_peronii_selection_tables"
# captures_labelled = [x.split('.')[0] for x in os.listdir(audio_dir)]

# # Generate pandas dataframe
# info_table = []
# for capture_id in captures_labelled:
#     info_table.append({
#         'fn': capture_id,
#         'audio_fp': os.path.join(audio_dir, f'{capture_id}.wav'),
#         'selection_table_fp': os.path.join(selection_table_dir, f'{capture_id}_selection_table.txt')
#     })

# info_df = pd.DataFrame.from_dict(info_table)

# # Export as CSV
# csv_out = 'datasets/frogid/Lim_peronii_info.csv'
# info_df.to_csv(csv_out)
# print(f'Saved info table to: {csv_out}')

Saved info table to: datasets/frogid/Lim_peronii_info.csv


# Option 2: Raw Raven Export

In [55]:
import os
import pandas as pd

# input_dir = "/home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_caerulea_formatted/Lit_caerulea_selection_tables"
# input_dir = "/home/admin/julia-dev/data/frogid_captures/strongly_labelled/Lim_peronii/selection_tables"
input_dir = "/home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables"

template_suffix = ".Table.1.selections.txt"

for filename in os.listdir(input_dir):
    if filename.endswith(template_suffix):
        
        full_path = os.path.join(input_dir, filename)
        print(f"Processing: {filename}")

        try:
            # Load Raven selection table (tab-separated)
            df = pd.read_csv(full_path, sep="\t")

            # Keep only required columns
            df_clean = df[["Begin Time (s)", "End Time (s)"]].copy()

            # Remove duplicate rows (waveform/spectrogram duplicates)
            df_clean = df_clean.drop_duplicates()

            # Add Annotation column
            df_clean["Annotation"] = "event"

            # Save cleaned file
            output_name = filename.replace(template_suffix, "_selection_table.txt")
            output_path = os.path.join(input_dir, output_name)

            df_clean.to_csv(output_path, sep="\t", index=False)

            print(f"Saved: {output_path}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

Processing: 722191.Table.1.selections.txt
Saved: /home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables/722191_selection_table.txt
Processing: 170439.Table.1.selections.txt
Saved: /home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables/170439_selection_table.txt
Processing: 96835.Table.1.selections.txt
Saved: /home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables/96835_selection_table.txt
Processing: 205288.Table.1.selections.txt
Saved: /home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables/205288_selection_table.txt
Processing: 156562.Table.1.selections.txt
Saved: /home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_peronii_selection_tables/156562_selection_table.txt
Processing: 290240.Table.1.selections.txt
Saved: /home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_peronii_formatted/Lit_pero

# Split into Train/Test/Val

In [6]:
INPUT_CSV = 'datasets/frogid/Lim_peronii_info.csv'
OUTPUT_DIR = '/home/admin/julia-dev/voxaboxen/datasets/frogid/Lim_peronii_formatted'
RANDOM_SEED = 42

TRAIN_RATIO = 0.80
TEST_RATIO  = 0.10
VAL_RATIO   = 1.0 - (TRAIN_RATIO + TEST_RATIO)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load CSV and treat first column as index
df = pd.read_csv(INPUT_CSV, index_col=0)
df.index.name = "index"

# Shuffle WITHOUT resetting index
df_shuf = df.sample(frac=1.0, random_state=RANDOM_SEED)

# Split
n = len(df_shuf)
n_train = int(n * TRAIN_RATIO)
n_test  = int(n * TEST_RATIO)

train_df = df_shuf.iloc[:n_train]
test_df  = df_shuf.iloc[n_train:n_train + n_test]
val_df   = df_shuf.iloc[n_train + n_test:]

# Write CSVs — index preserved, no extra column
train_df.to_csv(os.path.join(OUTPUT_DIR, "train_info.csv"),
                index=True, index_label=None)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_info.csv"),
               index=True, index_label=None)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_info.csv"),
              index=True, index_label=None)

print(f"Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")


Split sizes: train=41, val=6, test=5


# Download Audio

In [11]:
# Load captures table
captures_path = "/home/admin/julia-dev/data/frogid_captures/data_tables/FrogID_Captures_2025-03-09T21-01-13+0000.csv"
captures_df = pd.read_csv(captures_path)

/tmp/ipykernel_3447833/3616435119.py:3: DtypeWarning: Columns (32,34,35,39) have mixed types. Specify dtype option on import or set low_memory=False.
  captures_df = pd.read_csv(captures_path)


In [15]:
# Download audio files
def download_audio(audio_dir, capture_id, audio_url):
    save_path = os.path.join(audio_dir, f'{capture_id}.aac')
    urlretrieve(audio_url, save_path)
    audio = AudioSegment.from_file(save_path)
    wav_path = os.path.join(audio_dir, f'capture_{capture_id}.wav')
    audio.export(wav_path, format='wav')
    os.remove(save_path)

In [ ]:
CSV_PATH = "/home/admin/julia-dev/data/frogid_captures/data_tables/fraser_cuecounts_all_data.csv"
AUDIO_DIR = "/home/admin/julia-dev/data/frogid_captures/individual_count"

df = pd.read_csv(CSV_PATH)
count = 0

for _, row in df.iterrows():
    
    capture_id = row['id']
    species = row['species']

    audio_subdir = os.path.join(AUDIO_DIR, '_'.join(species.split()).lower())
    audio_path = os.path.join(audio_subdir, f'capture_{capture_id}.wav')

    if os.path.exists(audio_path):
        continue
    else:
        try:
            url = captures_df[captures_df['id'] == capture_id]['call_audio'].values[0]
        except IndexError:
            print(f'{capture_id} not found in captures table --> SKIPPING')
        else:
            download_audio(audio_subdir, capture_id, url)

/home/admin/julia-dev/voxaboxen/datasets/frogid/Lit_caerulea_multispecies_test


In [ ]:
species_dfs = {
    species: group.copy()
    for species, group in df.groupby("species")
}

{'Limnodynastes peronii':       Unnamed: 0      id                species        lat         lng  \
 1029           1    8458  Limnodynastes peronii -34.438900  150.469000   
 1030           2    8988  Limnodynastes peronii -33.804395  151.173691   
 1031           3    9346  Limnodynastes peronii -33.223300  151.504000   
 1032           4    9581  Limnodynastes peronii -33.725997  151.152149   
 1033           5   10032  Limnodynastes peronii -33.083200  151.434000   
 ...          ...     ...                    ...        ...         ...   
 1480         452  795854  Limnodynastes peronii -36.814200  149.794000   
 1481         453  797510  Limnodynastes peronii -33.805396  151.191289   
 1482         454  799406  Limnodynastes peronii -33.721400  150.700000   
 1483         455  801041  Limnodynastes peronii -33.803944  151.189016   
 1484         456  803310  Limnodynastes peronii -32.987000  151.586000   
 
           year  eventhour  earcount  callcount   duration       temp  \


In [51]:
DATASET_DIR = "/home/admin/julia-dev/voxaboxen/datasets/frogid"

def make_audio_fp(species, id_):
    folder = '_'.join(str(species).split()).lower()
    return os.path.join(AUDIO_DIR, folder, f"capture_{id_}.wav")

for species, df in species_dfs.items():
    new_df = pd.DataFrame({
        "fn": df["id"]
    })

    new_df["audio_fp"] = [
        make_audio_fp(species, id_)
        for species, id_ in zip(df["species"], df["id"])
    ]

    # remove bad paths / malformed species / missing files
    new_df = new_df[
        new_df["audio_fp"].apply(os.path.exists)
    ].copy()

    # optional: convert paths to strings
    new_df["audio_fp"] = new_df["audio_fp"].astype(str)
    new_df = new_df.reset_index(drop=True)

    parts = species.split()
    save_dir = os.path.join(DATASET_DIR, f'{'_'.join((parts[0][:3], parts[1]))}_multispecies_test')
    save_path = os.path.join(save_dir, 'test_info.csv')
    new_df.to_csv(save_path)